# Heart Disease Prediction — MLOps Assignment I
## BITS Pilani | MLOps (S2-25_AMLCSZG523)

**Dataset:** UCI Heart Disease (Cleveland) — https://archive.ics.uci.edu/dataset/45/heart+disease  
**GitHub:** https://github.com/jyotichughgit/heart-disease-mlops

This notebook covers **Tasks 1-4**:
- **Task 1** — Data Acquisition & Exploratory Data Analysis
- **Task 2** — Feature Engineering & Model Development
- **Task 3** — Experiment Tracking (MLflow)
- **Task 4** — Model Packaging & Reproducibility

---
## Task 1 — Data Acquisition & Exploratory Data Analysis
### 1.1 Install & Import Dependencies

In [ ]:
%pip install -q ucimlrepo mlflow scikit-learn pandas numpy matplotlib seaborn

In [ ]:
import os
import json
import pickle
import warnings

import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
    ConfusionMatrixDisplay,
)
from sklearn.model_selection import (
    StratifiedKFold,
    cross_val_score,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
%matplotlib inline

SEED = 42
np.random.seed(SEED)

SCREENSHOTS = '../screenshots'
os.makedirs(SCREENSHOTS, exist_ok=True)

# Feature definitions — same as train.py
NUMERICAL_FEATURES   = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
CATEGORICAL_FEATURES = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']

print('All libraries imported successfully!')


### 1.2 Data Acquisition from UCI Repository

In [ ]:
from ucimlrepo import fetch_ucirepo

# Fetch Heart Disease dataset (UCI ID = 45)
heart_disease = fetch_ucirepo(id=45)

X = heart_disease.data.features
y = heart_disease.data.targets

print(f'Dataset      : {heart_disease.metadata.name}')
print(f'UCI ID       : {heart_disease.metadata.uci_id}')
print(f'Instances    : {heart_disease.metadata.num_instances}')
print(f'Features     : {heart_disease.metadata.num_features}')
print(f'\nSummary:\n{heart_disease.metadata.additional_info.summary}')

In [ ]:
# Combine into a single DataFrame
df = X.copy()
df['target'] = y.values

# Binarize target: 0=no disease, 1-4=disease → 1
df['target'] = (df['target'] > 0).astype(int)

print(f'Dataset shape  : {df.shape}')
print(f'\nColumn names   : {df.columns.tolist()}')
print(f'\nData types:\n{df.dtypes}')
df.head()

In [ ]:
# Variable info from UCI
print('Variable information from UCI ML Repository:\n')
print(heart_disease.variables)

### 1.3 Data Cleaning

In [ ]:
# Show missing values BEFORE cleaning
missing = df.isnull().sum()
missing_cols = missing[missing > 0]

print('Missing values per column:')
print(missing_cols if len(missing_cols) > 0 else 'None')
print(f'\nTotal missing: {missing.sum()}')

In [ ]:
# Fill missing values:
# Numeric columns   → median
# Categorical columns → mode
for col in df.columns:
    if df[col].isnull().sum() > 0:
        if df[col].dtype in ['float64', 'int64']:
            median_val = df[col].median()
            df[col] = df[col].fillna(median_val)
            print(f"  Filled '{col}' missing values with median: {median_val}")
        else:
            mode_val = df[col].mode()[0]
            df[col] = df[col].fillna(mode_val)
            print(f"  Filled '{col}' missing values with mode: {mode_val}")

# Ensure all columns are numeric
for col in df.columns:
    df[col] = pd.to_numeric(df[col], errors='coerce')

rows_before = len(df)
df = df.dropna()
df = df.reset_index(drop=True)

print(f'\nRows dropped after numeric coercion : {rows_before - len(df)}')
print(f'Final dataset shape                 : {df.shape}')
print(f'\nMissing values remaining            : {df.isnull().sum().sum()}')

In [ ]:
# Save cleaned dataset
df.to_csv('heart_disease_cleaned.csv', index=False)
print("Cleaned dataset saved to 'heart_disease_cleaned.csv'")
print('\nBasic statistics:')
df.describe().T.round(2)

### 1.4 EDA Visualizations
#### Plot 1 — Class Balance

In [ ]:
counts = df['target'].value_counts().sort_index()
labels = ['No Disease (0)', 'Disease (1)']
colors = ['#4C72B0', '#DD8452']

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

bars = axes[0].bar(labels, counts.values, color=colors, edgecolor='black', width=0.5)
axes[0].set_title('Class Count', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Number of Patients')
axes[0].set_ylim(0, max(counts.values) * 1.25)
for bar, val in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, val + 1,
                 str(val), ha='center', fontweight='bold', fontsize=12)

axes[1].pie(counts.values, labels=labels, autopct='%1.1f%%',
            colors=colors, startangle=90, explode=(0.05, 0.05))
axes[1].set_title('Class Balance (%)', fontsize=13, fontweight='bold')

plt.suptitle('Class Distribution — Heart Disease Dataset',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{SCREENSHOTS}/eda_01_class_balance.png', bbox_inches='tight')
plt.show()

print(f'Class 0 (No Disease): {counts[0]} ({counts[0]/len(df):.1%})')
print(f'Class 1 (Disease):    {counts[1]} ({counts[1]/len(df):.1%})')

#### Plot 2 — Feature Distributions by Target Class

In [ ]:
feature_cols = [c for c in df.columns if c != 'target']
n_cols = 4
n_rows = (len(feature_cols) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, n_rows * 4))
axes = axes.flatten()

for i, col in enumerate(feature_cols):
    ax = axes[i]
    df[df['target'] == 0][col].hist(ax=ax, bins=25, alpha=0.6,
                                     color='#4C72B0', label='No Disease')
    df[df['target'] == 1][col].hist(ax=ax, bins=25, alpha=0.6,
                                     color='#DD8452', label='Disease')
    ax.set_title(col, fontweight='bold')
    ax.legend(fontsize=8)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Feature Distributions by Target Class',
             fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(f'{SCREENSHOTS}/eda_02_feature_distributions.png',
            dpi=150, bbox_inches='tight')
plt.show()

#### Plot 3 — Correlation Heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(14, 10))
corr = df.corr(numeric_only=True)
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f',
            cmap='RdYlGn', center=0, linewidths=0.5,
            ax=ax, annot_kws={'size': 9})
ax.set_title('Feature Correlation Heatmap', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{SCREENSHOTS}/eda_03_correlation_heatmap.png', bbox_inches='tight')
plt.show()

# Print top correlations with target
target_corr = corr['target'].drop('target').sort_values(key=abs, ascending=False)
print('Top feature correlations with target:')
for feat, val in target_corr.items():
    sign = '+' if val >= 0 else ''
    print(f'  {feat:<20}: {sign}{val:.3f}')

#### Plot 4 — Boxplots by Target Class

In [ ]:
continuous = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
fig, axes = plt.subplots(1, len(continuous), figsize=(18, 6))

for ax, col in zip(axes, continuous):
    data0 = df[df['target'] == 0][col]
    data1 = df[df['target'] == 1][col]
    bp = ax.boxplot([data0, data1], patch_artist=True,
                    labels=['No Disease', 'Disease'],
                    medianprops=dict(color='black', linewidth=2))
    bp['boxes'][0].set_facecolor('#4C72B0')
    bp['boxes'][1].set_facecolor('#DD8452')
    ax.set_title(col, fontweight='bold')
    ax.grid(True, alpha=0.3)

plt.suptitle('Continuous Feature Distribution by Class',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{SCREENSHOTS}/eda_04_boxplots.png', bbox_inches='tight')
plt.show()

---
## Task 2 — Feature Engineering & Model Development
### 2.1 Feature Preparation & Train/Test Split

In [ ]:
feature_cols = [c for c in df.columns if c != 'target']
X = df[feature_cols]
y = df['target']

print(f'Features shape : {X.shape}')
print(f'Target shape   : {y.shape}')
print(f'Target distribution:\n{y.value_counts().rename("target").to_frame()}')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)
print(f'\nTrain set : {len(X_train)} samples')
print(f'Test set  : {len(X_test)} samples')

### 2.2 Feature Scaling

In [ ]:
# Build preprocessing pipeline — same as train.py
# Numerical:   SimpleImputer(median)        + StandardScaler
# Categorical: SimpleImputer(most_frequent) + OneHotEncoder

def build_preprocessor():
    return ColumnTransformer([
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler',  StandardScaler()),
        ]), NUMERICAL_FEATURES),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
        ]), CATEGORICAL_FEATURES),
    ])

# Verify preprocessor builds correctly
test_pre = build_preprocessor()
X_test_pre = test_pre.fit_transform(X_train)
print(f'Preprocessing pipeline built successfully.')
print(f'Output shape after preprocessing: {X_test_pre.shape}')
print(f'Numerical features  : {NUMERICAL_FEATURES}')
print(f'Categorical features: {CATEGORICAL_FEATURES}')


### 2.3 Cross Validation

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
scoring = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']

# Models use Pipeline (preprocessor + classifier) — same as train.py
# Hyperparameters match train.py exactly
models = {
    'Logistic Regression': Pipeline([
        ('preprocessor', build_preprocessor()),
        ('classifier', LogisticRegression(C=1.0, max_iter=1000, random_state=SEED)),
    ]),
    'Random Forest': Pipeline([
        ('preprocessor', build_preprocessor()),
        ('classifier', RandomForestClassifier(n_estimators=200, max_depth=8, random_state=SEED)),
    ]),
    'Gradient Boosting': Pipeline([
        ('preprocessor', build_preprocessor()),
        ('classifier', GradientBoostingClassifier(n_estimators=150, learning_rate=0.1, max_depth=4, random_state=SEED)),
    ]),
}

cv_results = {}
for name, model in models.items():
    cv_results[name] = {}
    for metric in scoring:
        # CV on raw X_train — pipeline handles preprocessing internally
        scores = cross_val_score(model, X_train, y_train, cv=cv, scoring=metric)
        cv_results[name][metric] = (scores.mean(), scores.std())

# Display CV results
for name, metrics in cv_results.items():
    print(f'\n{name}:')
    for metric, (mean, std) in metrics.items():
        print(f'  {metric:<12}: {mean:.4f} (+/- {std:.4f})')


### 2.4 CV Comparison Bar Chart

In [ ]:
metrics_to_plot = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
model_names = list(cv_results.keys())
x = np.arange(len(metrics_to_plot))
width = 0.25
colors_bar = ['#3498db', '#2ecc71', '#e74c3c']

fig, ax = plt.subplots(figsize=(14, 6))
for i, (name, color) in enumerate(zip(model_names, colors_bar)):
    means = [cv_results[name][m][0] for m in metrics_to_plot]
    stds  = [cv_results[name][m][1] for m in metrics_to_plot]
    ax.bar(x + i * width, means, width, label=name,
           color=color, alpha=0.85, edgecolor='white', yerr=stds, capsize=4)

ax.set_xticks(x + width)
ax.set_xticklabels(metrics_to_plot, fontsize=11)
ax.set_ylabel('Score')
ax.set_ylim(0.5, 1.05)
ax.set_title('Cross Validation Scores — Model Comparison',
             fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(f'{SCREENSHOTS}/task2_cv_comparison.png', bbox_inches='tight')
plt.show()

### 2.5 Train Models & Classification Reports

In [ ]:
test_results = {}
for name, model in models.items():
    # Fit on raw X_train — pipeline handles preprocessing internally
    model.fit(X_train, y_train)
    y_pred  = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    test_results[name] = {
        'model':     model,
        'y_pred':    y_pred,
        'y_proba':   y_proba,
        'accuracy':  accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall':    recall_score(y_test, y_pred),
        'f1':        f1_score(y_test, y_pred),
        'roc_auc':   roc_auc_score(y_test, y_proba),
    }

    print(f'\n{"-"*50}')
    print(f'{name} — Test Set Results')
    print(f'{"-"*50}')
    print(classification_report(
        y_test, y_pred,
        target_names=['No Disease', 'Disease']
    ))
    for metric in ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']:
        print(f'  {metric:<12}: {test_results[name][metric]:.4f}')


### 2.6 ROC Curves

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
colors_roc = ['#3498db', '#2ecc71', '#e74c3c']

for (name, r), color in zip(test_results.items(), colors_roc):
    fpr, tpr, _ = roc_curve(y_test, r['y_proba'])
    ax.plot(fpr, tpr, color=color, lw=2.5,
            label=f'{name} (AUC={r["roc_auc"]:.3f})')

ax.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Random Classifier')
ax.set_xlabel('False Positive Rate', fontsize=13)
ax.set_ylabel('True Positive Rate', fontsize=13)
ax.set_title('ROC Curves — Model Comparison', fontsize=15, fontweight='bold')
ax.legend(loc='lower right', fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{SCREENSHOTS}/roc_curves.png', bbox_inches='tight')
plt.show()

### 2.7 Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
cmaps = ['Blues', 'Greens', 'Reds']

for ax, (name, r), cmap in zip(axes, test_results.items(), cmaps):
    ConfusionMatrixDisplay.from_predictions(
        y_test, r['y_pred'], ax=ax,
        display_labels=['No Disease', 'Disease'],
        colorbar=False, cmap=cmap
    )
    ax.set_title(name, fontweight='bold', fontsize=12)

plt.suptitle('Confusion Matrices — All Models',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{SCREENSHOTS}/confusion_matrices.png', bbox_inches='tight')
plt.show()

### 2.8 Best Model Summary

In [ ]:
best_model_name = max(test_results, key=lambda n: test_results[n]['roc_auc'])
best = test_results[best_model_name]

print(f'Best model: {best_model_name}')
print(f'\nTest set metrics:')
for metric in ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']:
    print(f'  {metric:<12}: {best[metric]:.4f}')

---
## Task 3 — Experiment Tracking with MLflow
### 3.1 Setup MLflow

In [ ]:
mlflow.set_tracking_uri('file:../mlruns')
mlflow.set_experiment('Heart_Disease_Classification')

print(f'MLflow tracking URI: {mlflow.get_tracking_uri()}')
print(f'Experiment: Heart_Disease_Classification')

### 3.2 Log All Model Runs

In [ ]:
for name, r in test_results.items():
    with mlflow.start_run(run_name=name):
        # Log all model hyperparameters
        params = r['model'].get_params()
        for param_name, param_value in params.items():
            try:
                mlflow.log_param(param_name, param_value)
            except Exception:
                pass

        # Log test metrics
        mlflow.log_metric('accuracy',  r['accuracy'])
        mlflow.log_metric('precision', r['precision'])
        mlflow.log_metric('recall',    r['recall'])
        mlflow.log_metric('f1_score',  r['f1'])
        mlflow.log_metric('roc_auc',   r['roc_auc'])

        # Log CV metrics
        for metric, (mean, std) in cv_results[name].items():
            mlflow.log_metric(f'cv_{metric}_mean', mean)
            mlflow.log_metric(f'cv_{metric}_std',  std)

        # Log model
        mlflow.sklearn.log_model(r['model'], artifact_path='model')

        # Log existing plot files as artifacts
        for plot_file in [f'{SCREENSHOTS}/roc_curves.png',
                          f'{SCREENSHOTS}/confusion_matrices.png',
                          f'{SCREENSHOTS}/task2_cv_comparison.png']:
            if os.path.exists(plot_file):
                mlflow.log_artifact(plot_file)

        print(f'Logged {name} -> AUC: {r["roc_auc"]:.4f}')

print('\nAll experiments logged to MLflow!')
print("Run 'mlflow ui' in the project directory to view the dashboard.")

---
## Task 4 — Model Packaging & Reproducibility
### 4.1 Build Full Pipeline for Best Model

In [ ]:
# Rebuild best model as a single sklearn Pipeline
# (preprocessor + classifier — ensures full reproducibility)
# Hyperparameters match train.py exactly
if best_model_name == 'Logistic Regression':
    final_estimator = LogisticRegression(C=1.0, max_iter=1000, random_state=SEED)
elif best_model_name == 'Random Forest':
    final_estimator = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=SEED)
else:
    final_estimator = GradientBoostingClassifier(n_estimators=150, learning_rate=0.1, max_depth=4, random_state=SEED)

pipeline = Pipeline([
    ('preprocessor', build_preprocessor()),
    ('classifier',   final_estimator),
])

# Fit on raw X_train — pipeline handles preprocessing internally
pipeline.fit(X_train, y_train)

y_pred_pipeline  = pipeline.predict(X_test)
y_proba_pipeline = pipeline.predict_proba(X_test)[:, 1]

print(f'Pipeline test accuracy : {accuracy_score(y_test, y_pred_pipeline):.4f}')
print(f'Pipeline test ROC-AUC  : {roc_auc_score(y_test, y_proba_pipeline):.4f}')


### 4.2 Save Pipeline & Metadata

In [ ]:
# Save pipeline as pickle
with open('model_pipeline.pkl', 'wb') as f:
    pickle.dump(pipeline, f)
print("Model pipeline saved to 'model_pipeline.pkl'")

# Save metadata
model_metadata = {
    'feature_names':    feature_cols,
    'target_name':      'target',
    'model_type':       best_model_name,
    'training_samples': len(X_train),
    'test_accuracy':    float(accuracy_score(y_test, y_pred_pipeline)),
    'test_roc_auc':     float(roc_auc_score(y_test, y_proba_pipeline)),
}
with open('model_metadata.json', 'w') as f:
    json.dump(model_metadata, f, indent=2)
print("Model metadata saved to 'model_metadata.json'")

# Verify — load and predict on 3 test samples
with open('model_pipeline.pkl', 'rb') as f:
    loaded_pipeline = pickle.load(f)

sample   = X_test.iloc[:3]
preds    = loaded_pipeline.predict(sample)
probs    = loaded_pipeline.predict_proba(sample)[:, 1]
actuals  = y_test.iloc[:3].tolist()

print(f'\nVerification — predictions on 3 test samples:')
print(f'  Predictions  : {preds.tolist()}')
print(f'  Probabilities: {[round(p, 3) for p in probs.tolist()]}')
print(f'  Actual       : {actuals}')

### 4.3 Log Best Model Pipeline to MLflow

In [ ]:
with mlflow.start_run(run_name=f'Best_Model_{best_model_name}'):
    mlflow.log_param('model_type', best_model_name)
    mlflow.log_param('pipeline', 'ColumnTransformer + Classifier')
    mlflow.log_metric('test_accuracy', accuracy_score(y_test, y_pred_pipeline))
    mlflow.log_metric('test_roc_auc',  roc_auc_score(y_test, y_proba_pipeline))
    mlflow.sklearn.log_model(pipeline, artifact_path='pipeline')
    mlflow.log_artifact('model_pipeline.pkl')
    mlflow.log_artifact('model_metadata.json')
    mlflow.log_artifact('heart_disease_cleaned.csv')

print('Best model pipeline logged to MLflow!')

---
## Summary

| Task | Status | Key Result |
|------|--------|------------|
| **Task 1 — EDA** | Done | 303 samples, 14 features, missing values filled (ca: median=0.0, thal: median=3.0) |
| **Task 2 — Model Development** | Done | Best: Logistic Regression (ROC-AUC=0.9513) |
| **Task 3 — Experiment Tracking** | Done | 3 MLflow runs + best model run logged |
| **Task 4 — Model Packaging** | Done | Pipeline saved as pickle, verified on 3 test samples |

**GitHub:** https://github.com/jyotichughgit/heart-disease-mlops  
**Deployed API:** http://34.31.48.169:8000  
**Swagger UI:** http://34.31.48.169:8000/docs